## Data processing with the standard python library

In [63]:
print("##########################################################")
print("""
Use standard python libraries to do the transformations
"""
)
print("##########################################################")

##########################################################

Use standard python libraries to do the transformations

##########################################################


In [64]:
import csv

#reading the data from the csv file into a list of dicts called data
data = []
with open("sample_data.csv","r", newline= "") as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        data.append(row)
print(data[:3])

[{'Customer_ID': '1', 'Customer_Name': 'Henry Jones', 'Age': '32', 'Gender': 'Male', 'Purchase_Amount': '1080000.66', 'Purchase_Date': '2023-08-15'}, {'Customer_ID': '2', 'Customer_Name': 'Emma Rodriguez', 'Age': '24', 'Gender': 'Male', 'Purchase_Amount': '62.4', 'Purchase_Date': '2024-04-16'}, {'Customer_ID': '3', 'Customer_Name': 'Frank Martinez', 'Age': '20', 'Gender': 'Female', 'Purchase_Amount': '443.47', 'Purchase_Date': '2024-05-16'}]


In [65]:
# remove duplicate records based on customer id using sets

data_unique = []
customer_ids_seen = set()
for row in data:
    if row['Customer_ID'] not in customer_ids_seen:
        data_unique.append(row)
        customer_ids_seen.add(row['Customer_ID'])
    else:
        print(f'duplicate customer id {row["Customer_ID"]}')



duplicate customer id 84
duplicate customer id 85
duplicate customer id 86
duplicate customer id 87
duplicate customer id 88
duplicate customer id 89
duplicate customer id 90
duplicate customer id 91


In [66]:
# if an entry in data_unique does not have Age then set it to 0.
# do the same for purchase_amount

for row in data_unique:
    if not row['Age']:
        print(f"Customer {row['Customer_Name']} does not have Age value")
        row['Age'] =0
    if not row['Purchase_Amount']:
        print(f"Customer {row['Customer_Name']} does not have a purchase amount value")
        row['Purchase_Amount'] = 0

# remove outliers, assuming we define the outliers as any record with an age over 100 or under 0
data_cleaned = [
    row for row in data_unique if int(row['Age']) <=100 and int(row['Age'])>=0
]

# convert the gender column to a binary forma (0 for female, 1 for male)
for row in data_cleaned:
    if row['Gender']== "Female":
        row['Gender'] = 0
    elif row['Gender'] == "Male":
        row['Gender'] = 1

#split the customer name dtaa into separate first name and last name
for row in data_cleaned:
    first_name, last_name = row['Customer_Name'].split(" ", maxsplit= 1)
    row['First_Name']= first_name
    row['Last_Name']= last_name

print(data_cleaned[:3])

Customer Alice Johnson does not have Age value
Customer Jack Garcia does not have Age value
[{'Customer_ID': '1', 'Customer_Name': 'Henry Jones', 'Age': '32', 'Gender': 1, 'Purchase_Amount': '1080000.66', 'Purchase_Date': '2023-08-15', 'First_Name': 'Henry', 'Last_Name': 'Jones'}, {'Customer_ID': '2', 'Customer_Name': 'Emma Rodriguez', 'Age': '24', 'Gender': 1, 'Purchase_Amount': '62.4', 'Purchase_Date': '2024-04-16', 'First_Name': 'Emma', 'Last_Name': 'Rodriguez'}, {'Customer_ID': '3', 'Customer_Name': 'Frank Martinez', 'Age': '20', 'Gender': 0, 'Purchase_Amount': '443.47', 'Purchase_Date': '2024-05-16', 'First_Name': 'Frank', 'Last_Name': 'Martinez'}]


In [67]:
# calculating the total purchase amount for each Gender
from collections import defaultdict
total_purchase_by_gender = defaultdict(float)

for row in data_cleaned:
    total_purchase_by_gender[row['Gender']] += float(row['Purchase_Amount'])

#calculate the average purchase amount by age group as defined below
age_groups = {'18-30':[], '31-40':[], '41-50':[], '51-60':[], '61-70':[]}

for row in data_cleaned:
    age = int(row['Age'])
    if age <= 30:
        age_groups['18-30'].append(float(row['Purchase_Amount']))
    elif age <= 40:
        age_groups['31-40'].append(float(row['Purchase_Amount']))
    if age <= 50:
        age_groups['41-50'].append(float(row['Purchase_Amount']))
    if age <= 60:
        age_groups['51-60'].append(float(row['Purchase_Amount']))
    else:
        age_groups['61-70'].append(float(row['Purchase_Amount']))


average_purchase_by_age_group = {
    group:sum(amounts)/ len(amounts) for group, amounts in age_groups.items()
}

print("Total purchase amount by Gender:", total_purchase_by_gender)
print("Average purchase amount by Age Group:", average_purchase_by_age_group)

Total purchase amount by Gender: defaultdict(<class 'float'>, {1: 1104600.549999999, 0: 28215.780000000002})
Average purchase amount by Age Group: {'18-30': 567.9048387096775, '31-40': 60524.432222222225, '41-50': 16187.300724637667, '51-60': 13085.239186046501, '61-70': 534.6971428571428}


## Data Processing with PySpark

In [68]:
print("##########################################################")
print("""Use PySpark DataFrame API to do the transformations""")
print("##########################################################")

##########################################################
Use PySpark DataFrame API to do the transformations
##########################################################


In [69]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, lit, when, split, sum as spark_sum, avg, regexp_replace
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

spark = SparkSession.builder.appName('CSV Data Processing').getOrCreate()

schema = StructType([
    StructField("Customer_ID", IntegerType(), True),
    StructField("Customer_name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Purchase_Amount", FloatType(), True),
    StructField("Purchase_Date", DateType(), True)
])

26/04/07 23:04:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [70]:
# reading data from csv file into the DataFrame
data = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .schema(schema) \
    .csv("./sample_data.csv")

In [71]:
# Handling duplicate rows
data_unique = data.dropDuplicates()

In [72]:
# Handling missing values by replacing them with 0 in pyspark
data_cleaned_missing = data_unique.select(
    col("Customer_ID"),
    col("Customer_Name"),
    coalesce(col("Age"),lit(0)).alias("Age"),
    col("Gender"),
    coalesce(col("Purchase_Amount"),lit(0.0)).alias("Purchase_Amount"),
    col("Purchase_Date")
)

In [73]:
# Removing outliers
data_cleaned_outliers = data_cleaned_missing.filter(
    (col("Age")<= 100) & (col("Age")>0)
)

In [74]:
# Converting the gender column to a binary value
data_cleaned_gender = data_cleaned_outliers.withColumn(
    "Gender_Binary",
    when(col("Gender")== "Female",0).otherwise(1)
)

In [75]:
# Separating customer name into first and last name columns
data_cleaned = data_cleaned_gender.select(
    col("Customer_ID"),
    split(col("Customer_Name"), " ").getItem(0).alias("First_Name"),
    split(col("Customer_Name"), " ").getItem(1).alias("Last_Name"),
    col("Age"),
    col("Gender_Binary"),
    col("Purchase_Amount"),
    col("Purchase_Date")
)

In [76]:
total_purchase_by_gender = data_cleaned.groupBy("Gender_Binary") \
    .agg(spark_sum("Purchase_Amount").alias("Total_Purchase_Amount")) \
        .collect()

In [77]:
# Calculate the average purchase amount by age group
average_purchase_by_age_group = data_cleaned.withColumn(
    "Age_Group",
    when((col("Age")>= 18) & (col("Age")<=30), "18-30")
    .when((col("Age")>= 31) & (col("Age")<=40), "31-40")
    .when((col("Age")>= 41) & (col("Age")<=50), "41-50")
    .when((col("Age")>= 51) & (col("Age")<=60), "51-60")
    .otherwise("61-70")
).groupby("Age_Group") \
    .agg(avg("Purchase_Amount").alias("Average_Purchase_Amount")) \
    .collect()

In [78]:
# printing out the results
print("====================== Results ==========================")
print("Total purchase amount by Gender:")
for row in total_purchase_by_gender:
    print(f"Gender_Binary: {row['Gender_Binary']}, Total_Purchase_Amount: {row['Total_Purchase_Amount']}")

print("Average purchase amount by Age group:")
for row in average_purchase_by_age_group:
    print(f"Age_Group: {row['Age_Group']}, Average_Purchase_Amount: {row['Average_Purchase_Amount']}")

====================== Results ==========================
Total purchase amount by Gender:
Gender_Binary: 1, Total_Purchase_Amount: 1104600.5150527954
Gender_Binary: 0, Total_Purchase_Amount: 27164.30993461609
Average purchase amount by Age group:
Age_Group: 18-30, Average_Purchase_Amount: 570.8131050899111
Age_Group: 41-50, Average_Purchase_Amount: 493.946000289917
Age_Group: 31-40, Average_Purchase_Amount: 60524.43027750651
Age_Group: 51-60, Average_Purchase_Amount: 494.51882250168745
Age_Group: 61-70, Average_Purchase_Amount: 534.6971397399902


In [79]:
data_cleaned.show(10)

+-----------+----------+---------+---+-------------+------------------+-------------+
|Customer_ID|First_Name|Last_Name|Age|Gender_Binary|   Purchase_Amount|Purchase_Date|
+-----------+----------+---------+---+-------------+------------------+-------------+
|         68|     Frank|   Miller| 34|            0| 873.3599853515625|   2024-01-27|
|          3|     Frank| Martinez| 20|            0| 443.4700012207031|   2024-05-16|
|         80|   Charlie|    Jones| 38|            0| 282.3500061035156|   2024-04-14|
|         21|     Grace|Rodriguez| 21|            0|389.69000244140625|   2023-07-19|
|         39|     Grace|   Garcia| 23|            1| 901.8200073242188|   2023-12-09|
|         22|       Ivy|    Brown| 34|            1|             697.0|   2023-07-15|
|         52|     Frank|  Johnson| 54|            0| 281.3500061035156|   2023-07-02|
|         67|     Frank|  Johnson| 60|            0|  335.989990234375|   2023-10-03|
|         58|     Alice|    Jones| 50|            1| 3

## Exercises 

In [ ]:
# 1 
total_purchase_by_gender = data_cleaned.groupBy("Gender_Binary") \
    .agg(spark_sum("Purchase_Amount").alias("Total_Purchase_Amount")) \
        .collect()

print(total_purchase_by_gender)

[Row(Gender_Binary=1, Total_Purchase_Amount=1104600.5150527954), Row(Gender_Binary=0, Total_Purchase_Amount=27164.30993461609)]


In [84]:
# 2
# Calculate the average purchase amount by age group
average_purchase_by_age_group = data_cleaned.withColumn(
    "Age_Group",
    when((col("Age")>= 18) & (col("Age")<=30), "18-30")
    .when((col("Age")>= 31) & (col("Age")<=40), "31-40")
    .when((col("Age")>= 41) & (col("Age")<=50), "41-50")
    .when((col("Age")>= 51) & (col("Age")<=60), "51-60")
    .otherwise("61-70")
).groupby("Age_Group") \
    .agg(avg("Purchase_Amount").alias("Average_Purchase_Amount")) \
    .collect()

print(average_purchase_by_age_group)

[Row(Age_Group='18-30', Average_Purchase_Amount=570.8131050899111), Row(Age_Group='41-50', Average_Purchase_Amount=493.946000289917), Row(Age_Group='31-40', Average_Purchase_Amount=60524.43027750651), Row(Age_Group='51-60', Average_Purchase_Amount=494.51882250168745), Row(Age_Group='61-70', Average_Purchase_Amount=534.6971397399902)]


In [85]:
spark.stop()